<a href="https://colab.research.google.com/github/ks148678-beep/Data-Warehousing-and-Data-Mining/blob/main/Loan_cleaned_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
"""
Loan Data Cleaning Script

Dataset: loan_train_1.xls (actually a quoted CSV, not a real .xls file)
Raw file problem: Every row is wrapped in " " quotes, so Excel / pandas
reads it as 1 column instead of 10 columns.
"""

'\nLoan Data Cleaning Script\n\nDataset: loan_train_1.xls (actually a quoted CSV, not a real .xls file)\nRaw file problem: Every row is wrapped in " " quotes, so Excel / pandas\nreads it as 1 column instead of 10 columns.\n'

In [14]:
import pandas as pd
import numpy as np
from pathlib import Path

# --------------------------------------------------
# 1. SET PATHS
# --------------------------------------------------
# Updated to match the actual file location in your environment
INPUT_PATH = Path("/content/loan_train (1).xls")
OUTPUT_PATH = Path("/content/loan_train_cleaned.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 2. LOAD RAW FILE - FIX QUOTED ROWS
# --------------------------------------------------
# The file looks like this:
# "0,0,PAIDOFF,1000,30,9/8/2016,10/7/2016,45,High School or Below,male"
# We must strip the outer quotes before splitting by comma.

print(f"Loading raw file: {INPUT_PATH}")

with open(INPUT_PATH, 'r', encoding='utf-8', errors='ignore') as f:
    lines = f.readlines()

print(f"Raw lines found: {len(lines)}")

# Remove outer quotes: "...." -> ....
clean_lines = []
for line in lines:
    s = line.strip()
    if s.startswith('"') and s.endswith('"'):
        s = s[1:-1]
    clean_lines.append(s)

# First line is header, rest is data
header = clean_lines[0].split(',')
data = [line.split(',') for line in clean_lines[1:]]

print(f"Header found: {header}")
print(f"Data rows found: {len(data)}")

df_raw = pd.DataFrame(data, columns=header)

print("\n--- RAW SHAPE ---")
print(f"Raw shape: {df_raw.shape}")
print(df_raw.head(3).to_string())

# --------------------------------------------------

Loading raw file: /content/loan_train (1).xls
Raw lines found: 347
Header found: ['', 'Unnamed: 0', 'loan_status', 'Principal', 'terms', 'effective_date', 'due_date', 'age', 'education', 'Gender']
Data rows found: 346

--- RAW SHAPE ---
Raw shape: (346, 10)
     Unnamed: 0 loan_status Principal terms effective_date   due_date age             education  Gender
0  0          0     PAIDOFF      1000    30       9/8/2016  10/7/2016  45  High School or Below    male
1  2          2     PAIDOFF      1000    30       9/8/2016  10/7/2016  33              Bechalor  female
2  3          3     PAIDOFF      1000    15       9/8/2016  9/22/2016  27               college    male


In [15]:
# 3. CLEAN COLUMN NAMES AND DROP DUPLICATE INDEX
# --------------------------------------------------
# Raw has 2 identical index columns:
# Column 0 = '' (blank name) and Column 1 = 'Unnamed: 0'
# Both contain the same Loan_ID. Keep one, drop the other.

print("\n--- CHECK DUPLICATE INDEX ---")
print(f"Index columns identical? {(df_raw.iloc[:, 0] == df_raw.iloc[:, 1]).all()}")

df = pd.DataFrame()
df['Loan_ID'] = df_raw['Unnamed: 0'].astype(int)
df['loan_status'] = df_raw['loan_status']
df['Principal'] = df_raw['Principal']
df['terms'] = df_raw['terms']
df['effective_date'] = df_raw['effective_date']
df['due_date'] = df_raw['due_date']
df['age'] = df_raw['age']
df['education'] = df_raw['education']
df['Gender'] = df_raw['Gender']

print(f"Shape after dropping duplicate index: {df.shape}")

# --------------------------------------------------


--- CHECK DUPLICATE INDEX ---
Index columns identical? True
Shape after dropping duplicate index: (346, 9)


In [16]:
# 4. DATA QUALITY CHECKS (BEFORE CLEANING)
# --------------------------------------------------
print("\n--- MISSING VALUES ---")
print(df.isna().sum())

print("\n--- DUPLICATED ROWS ---")
print(f"Fully duplicated rows: {df.duplicated().sum()}")

print("\n--- UNIQUE VALUES BEFORE CLEANING ---")
print(f"education: {df['education'].unique()}")
print(f"Gender: {df['Gender'].unique()}")
print(f"loan_status: {df['loan_status'].unique()}")
print(f"Principal: {sorted(df['Principal'].unique())}")
print(f"terms: {sorted(df['terms'].unique())}")

# Check missing Loan_IDs - gaps in sequence
loan_ids = set(df['Loan_ID'])
expected_ids = set(range(df['Loan_ID'].min(), df['Loan_ID'].max() + 1))
missing_ids = sorted(expected_ids - loan_ids)
print(f"\nLoan_ID range: {df['Loan_ID'].min()} to {df['Loan_ID'].max()}")
print(f"Missing Loan_IDs (n={len(missing_ids)}): {missing_ids[:20]}...")
print("NOTE: Do NOT renumber. These were removed in original train/test split.")

# --------------------------------------------------
# 5. STANDARDISE TEXT
# --------------------------------------------------
# Fix spelling: Bechalor -> Bachelor, college -> College
# Fix case: male -> Male, female -> Female

df['loan_status'] = df['loan_status'].str.strip().str.upper()

education_map = {
    'High School or Below': 'High School or Below',
    'college': 'College',
    'Bechalor': 'Bachelor',
    'Master or Above': 'Master or Above'
}
df['education'] = df['education'].str.strip().map(education_map)

df['Gender'] = df['Gender'].str.strip().str.title()

print("\n--- AFTER TEXT CLEANING ---")
print(df['education'].value_counts())
print(df['Gender'].value_counts())
print(df['loan_status'].value_counts())

# ----------------------------------


--- MISSING VALUES ---
Loan_ID           0
loan_status       0
Principal         0
terms             0
effective_date    0
due_date          0
age               0
education         0
Gender            0
dtype: int64

--- DUPLICATED ROWS ---
Fully duplicated rows: 0

--- UNIQUE VALUES BEFORE CLEANING ---
education: ['High School or Below' 'Bechalor' 'college' 'Master or Above']
Gender: ['male' 'female']
loan_status: ['PAIDOFF' 'COLLECTION']
Principal: ['1000', '300', '500', '800', '900']
terms: ['15', '30', '7']

Loan_ID range: 0 to 399
Missing Loan_IDs (n=54): [1, 5, 21, 24, 35, 37, 38, 48, 50, 61, 64, 68, 76, 78, 84, 85, 88, 91, 96, 100]...
NOTE: Do NOT renumber. These were removed in original train/test split.

--- AFTER TEXT CLEANING ---
education
High School or Below    151
College                 149
Bachelor                 44
Master or Above           2
Name: count, dtype: int64
Gender
Male      294
Female     52
Name: count, dtype: int64
loan_status
PAIDOFF       260
COLLECTIO

In [17]:
# 6. CONVERT DATA TYPES
# --------------------------------------------------
df['Principal'] = df['Principal'].astype(int)
df['terms'] = df['terms'].astype(int)
df['age'] = df['age'].astype(int)

# Dates were text like 9/8/2016 -> convert to real datetime
# Format is Month/Day/Year
df['effective_date'] = pd.to_datetime(df['effective_date'], format='%m/%d/%Y')
df['due_date'] = pd.to_datetime(df['due_date'], format='%m/%d/%Y')

print("\n--- DATA TYPES AFTER CONVERSION ---")
print(df.dtypes)

# Check age outliers
print("\n--- AGE CHECK ---")
print(df['age'].describe())
print(f"Age < 18 or > 80: {((df['age'] < 18) | (df['age'] > 80)).sum()} (0 = good)")

# --------------------------------------------------


--- DATA TYPES AFTER CONVERSION ---
Loan_ID                    int64
loan_status               object
Principal                  int64
terms                      int64
effective_date    datetime64[ns]
due_date          datetime64[ns]
age                        int64
education                 object
Gender                    object
dtype: object

--- AGE CHECK ---
count    346.000000
mean      30.939306
std        6.039418
min       18.000000
25%       27.000000
50%       30.000000
75%       35.000000
max       51.000000
Name: age, dtype: float64
Age < 18 or > 80: 0 (0 = good)


In [18]:
# 7. VALIDATE DUE_DATE AGAINST TERMS
# --------------------------------------------------
# Rule in this dataset: due_date = effective_date + terms - 1
# (inclusive counting: 9/8 + 30 days - 1 = 10/7)
# Most rows follow this. 24 rows break it - we flag them, not delete them.

df['expected_due_date'] = df['effective_date'] + pd.to_timedelta(df['terms'] - 1, unit='D')
df['due_date_error_flag'] = (df['due_date'] != df['expected_due_date']).astype(int)
df['loan_duration_days'] = (df['due_date'] - df['effective_date']).dt.days + 1

print("\n--- DUE DATE VALIDATION ---")
print(f"Rows with due_date error: {df['due_date_error_flag'].sum()} / {len(df)} ({df['due_date_error_flag'].mean()*100:.1f}%)")
print("Error breakdown by terms vs duration:")
print(df[df['due_date_error_flag'] == 1][['Loan_ID', 'effective_date', 'due_date', 'terms', 'loan_duration_days']].head(10).to_string())

# --------------------------------------------------


--- DUE DATE VALIDATION ---
Rows with due_date error: 24 / 346 (6.9%)
Error breakdown by terms vs duration:
     Loan_ID effective_date   due_date  terms  loan_duration_days
9         11     2016-09-10 2016-10-09     15                  30
35        42     2016-09-11 2016-09-24      7                  14
88       109     2016-09-11 2016-09-24      7                  14
90       111     2016-09-11 2016-11-09     30                  60
101      122     2016-09-11 2016-10-25     15                  45
116      137     2016-09-11 2016-10-25     15                  45
124      146     2016-09-11 2016-11-09     30                  60
140      165     2016-09-12 2016-11-10     30                  60
174      206     2016-09-12 2016-11-10     30                  60
183      215     2016-09-12 2016-11-10     30                  60


In [19]:
# 8. FEATURE ENGINEERING FOR ANALYSIS (Q1)
# --------------------------------------------------
# These features are needed for Question 1 and 2 and avoid leakage

# Weekend flag: 5 = Saturday, 6 = Sunday (Monday=0)
df['dayofweek'] = df['effective_date'].dt.dayofweek
df['day_name'] = df['effective_date'].dt.day_name()
df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)

print("\n--- WEEKEND CHECK ---")
print(df['day_name'].value_counts())
print(f"Weekend loans: {df['is_weekend'].sum()} ({df['is_weekend'].mean()*100:.1f}%)")

# --------------------------------------------------
# 9. FINAL CHECKS AND SAVE
# --------------------------------------------------
# Reorder columns for clean output
column_order = [
    'Loan_ID', 'loan_status', 'Principal', 'terms',
    'effective_date', 'due_date', 'expected_due_date',
    'due_date_error_flag', 'loan_duration_days',
    'age', 'education', 'Gender',
    'dayofweek', 'day_name', 'is_weekend'
]
df_clean = df[column_order].copy()

print("\n--- FINAL CLEANED DATA ---")
print(f"Final shape: {df_clean.shape}")
print(df_clean.head(5).to_string())
print(f"Missing values total: {df_clean.isna().sum().sum()}")

# Save
df_clean.to_csv(OUTPUT_PATH, index=False)
print(f"\nCleaned file saved to: {OUTPUT_PATH}")
print("Done!")


--- WEEKEND CHECK ---
day_name
Sunday       166
Monday        87
Saturday      31
Wednesday     31
Tuesday       20
Friday         8
Thursday       3
Name: count, dtype: int64
Weekend loans: 197 (56.9%)

--- FINAL CLEANED DATA ---
Final shape: (346, 15)
   Loan_ID loan_status  Principal  terms effective_date   due_date expected_due_date  due_date_error_flag  loan_duration_days  age             education  Gender  dayofweek  day_name  is_weekend
0        0     PAIDOFF       1000     30     2016-09-08 2016-10-07        2016-10-07                    0                  30   45  High School or Below    Male          3  Thursday           0
1        2     PAIDOFF       1000     30     2016-09-08 2016-10-07        2016-10-07                    0                  30   33              Bachelor  Female          3  Thursday           0
2        3     PAIDOFF       1000     15     2016-09-08 2016-09-22        2016-09-22                    0                  15   27               College    Male   

In [21]:
# Save whatever dataframe you have to Excel
# Change df_clean to the name of your dataframe

df_clean.to_excel("loan_cleaned.xlsx", index=False, sheet_name="Cleaned Data")
print("Saved as loan_cleaned.xlsx")

Saved as loan_cleaned.xlsx
